# Joint KT + Calibration on ASSISTments 2009

Joint LSTM model on `assistments_2009.csv` that predicts next-step correctness and calibration score. Dependencies: `pandas`, `numpy`, `torch`, `scikit-learn`.

In [9]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, mean_squared_error
from sklearn.model_selection import train_test_split
from IPython.display import Markdown, display

CSV_PATH = "assistments_2009.csv"
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


## 1. Load Dataset

In [10]:
def detect_column(df: pd.DataFrame, candidates: list[str], description: str) -> str:
    """Pick first column whose normalized name matches a candidate."""
    norm = {c: re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_") for c in df.columns}
    cand_norm = [re.sub(r"[^a-z0-9]+", "_", c.lower()).strip("_") for c in candidates]
    for want in cand_norm:
        for col, n in norm.items():
            if n == want or n.endswith("_" + want) or want in n.split("_"):
                return col
    raise ValueError(f"Could not detect column for {description}. Tried {candidates}. Got columns: {list(df.columns)}")


df = pd.read_csv(CSV_PATH, encoding='latin-1', low_memory=False)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")

col_user = detect_column(df, ["user_id", "userid", "student_id", "user"], "user_id")
col_skill_name = detect_column(df, ["skill_name", "skillname", "kc", "skill"], "skill_name")
col_correct = detect_column(df, ["correct", "is_correct", "label"], "correct")
col_order = detect_column(df, ["order_id", "orderid", "log_id", "row_id"], "order_id")

print("Detected columns:")
print(f"  user_id    -> {col_user!r}")
print(f"  skill_name -> {col_skill_name!r}")
print(f"  correct    -> {col_correct!r}")
print(f"  order_id   -> {col_order!r}")

Loaded 525,534 rows, 30 columns
Detected columns:
  user_id    -> 'user_id'
  skill_name -> 'skill_name'
  correct    -> 'correct'
  order_id   -> 'order_id'


## 2. Data Cleaning

In [11]:
work = df[[col_user, col_skill_name, col_correct, col_order]].copy()
work.columns = ["user_id", "skill_name", "correct", "order_id"]

work["correct"] = pd.to_numeric(work["correct"], errors="coerce")
before = len(work)
work = work.dropna(subset=["correct"])
print(f"Dropped null correctness: {before - len(work):,} rows")

counts = work.groupby("user_id").size()
keep_users = counts[counts >= 10].index
work = work[work["user_id"].isin(keep_users)].copy()
print(f"Removed users with <10 interactions; {len(work):,} rows remain")

work["order_id"] = pd.to_numeric(work["order_id"], errors="coerce")
work = work.dropna(subset=["order_id", "skill_name"])
work["skill_name"] = work["skill_name"].astype(str)
work["correct"] = work["correct"].astype(np.float32).clip(0, 1)

time_col = None
for c in df.columns:
    n = re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")
    if n in ("ms_first_response", "msfirstresponse", "response_time", "duration_ms"):
        time_col = c
        break
if time_col is not None:
    work["response_time_ms"] = pd.to_numeric(df.loc[work.index, time_col], errors="coerce")
else:
    work["response_time_ms"] = np.nan

print(work.head())

Dropped null correctness: 0 rows
Removed users with <10 interactions; 520,940 rows remain
   user_id       skill_name  correct  order_id  response_time_ms
0    64525  Box and Whisker      1.0  33022537             32454
1    64525  Box and Whisker      1.0  33022709              4922
2    70363  Box and Whisker      0.0  35450204             25390
3    70363  Box and Whisker      1.0  35450295              4859
4    70363  Box and Whisker      0.0  35450311             19813


## 3. Feature Engineering

In [12]:
work = work.sort_values(["user_id", "order_id"], kind="mergesort").reset_index(drop=True)

skill_codes, skill_uniques = pd.factorize(work["skill_name"], sort=True)
work["skill_id"] = skill_codes.astype(np.int64)
num_skills = int(work["skill_id"].max()) + 1
print(f"Unique skills (from skill_name): {num_skills}")

if work["response_time_ms"].notna().any():
    med = work["response_time_ms"].median()
    iqr = work["response_time_ms"].quantile(0.75) - work["response_time_ms"].quantile(0.25)
    scale = float(iqr) if iqr and iqr > 0 else float(work["response_time_ms"].std() or 1.0)
    work["response_time_norm"] = ((work["response_time_ms"] - med) / scale).clip(-5, 5).fillna(0.0)
    print(f"Normalized response time -> response_time_norm (raw column {time_col!r})")
else:
    work["response_time_norm"] = 0.0
    print("No usable response-time values; response_time_norm set to 0.")

work.head()

Unique skills (from skill_name): 110
Normalized response time -> response_time_norm (raw column 'ms_first_response')


,user_id,skill_name,correct,order_id,response_time_ms,skill_id,response_time_norm
0,14,Circle Graph,0.0,21617623,26271,18,0.082845
1,14,Percent Of,0.0,21617623,26271,67,0.082845
2,14,Circle Graph,1.0,21617632,29123,18,0.146138
3,14,Percent Of,1.0,21617632,29123,67,0.146138
4,14,Circle Graph,0.0,21617641,13779,18,-0.194385


## 4. Simulate Confidence

High-confidence distribution for correct responses **Beta(8, 2)**, low-confidence for incorrect **Beta(2, 8)**, plus small Gaussian noise (clipped to \[0, 1\]).

In [13]:
rng = np.random.default_rng(RANDOM_SEED)
correct_mask = work["correct"].to_numpy() >= 0.5
conf = np.empty(len(work), dtype=np.float64)
conf[correct_mask] = rng.beta(8, 2, size=correct_mask.sum())
conf[~correct_mask] = rng.beta(2, 8, size=(~correct_mask).sum())
noise = rng.normal(0, 0.05, size=len(work))
work["confidence"] = np.clip(conf + noise, 0.0, 1.0)
work[["correct", "confidence"]].describe()

,correct,confidence
count,442578.000000,442578.000000
mean,0.695604,0.617307
std,0.460152,0.304071
min,0.000000,0.000000
25%,0.000000,0.317685
50%,1.000000,0.735574
75%,1.000000,0.861524
max,1.000000,1.000000


### Methodology Note: Confidence Simulation

Because ASSISTments 2009 does not include self-reported confidence estimates, confidence scores were simulated using Beta distributions conditioned on correctness. Correct responses were assigned confidence values sampled from Beta(8, 2), while incorrect responses were assigned confidence values sampled from Beta(2, 8), with small Gaussian noise added to introduce variability. This procedure creates a dataset that is calibrated by design, as confidence is partially derived from correctness. Consequently, miscalibration events detected by the model arise primarily from the injected noise rather than authentic differences in learner self-assessment. While this limits the psychological realism of the calibration signal, it provides a controlled environment for evaluating the proposed framework and verifying that the model can detect and respond to confidence–performance discrepancies. Future work should validate the approach using datasets that contain real learner confidence ratings and naturally occurring metacognitive variation.

## 5. Train / Validation / Test Split

Split **by `user_id`**: 80% train, 10% validation, 10% test.

In [14]:
all_users = work["user_id"].unique()
train_users, temp_users = train_test_split(all_users, test_size=0.2, random_state=RANDOM_SEED)
val_users, test_users = train_test_split(temp_users, test_size=0.5, random_state=RANDOM_SEED)

train_df = work[work["user_id"].isin(train_users)]
val_df = work[work["user_id"].isin(val_users)]
test_df = work[work["user_id"].isin(test_users)]

print(f"Users: train={len(train_users):,}  val={len(val_users):,}  test={len(test_users):,}")
print(f"Rows:  train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

Users: train=2,560  val=320  test=321
Rows:  train=349,434  val=41,591  test=51,553


## 6. Joint KT + Calibration Model Definition

Per timestep, input is **concat(one-hot(skill), correct × one-hot(skill), confidence, response_time_norm)** → size **2 × num_skills + 2** (222 for ASSISTments). A shared 2-layer LSTM feeds two heads: KT (sigmoid) and calibration (tanh).

### Input Design: Confidence as a Third Channel

Confidence is included as an additional input feature at each timestep so the LSTM can condition its hidden state on the learner's self-reported certainty alongside skill identity and prior performance. This allows the shared representation to capture interactions between knowledge state and metacognitive signals—e.g., whether a student is overconfident on a skill they have not yet mastered—rather than treating calibration as an isolated post-hoc prediction.

### Input Design: Response Time as a Fourth Channel

Response time is included as a fourth input feature, as prior work has shown it to be informative of both knowledge state and cognitive load.

### Calibration Target: confidence[t] − correct[t]

The calibration head is trained to predict **confidence[t] − correct[t]**, the signed gap between self-reported confidence and actual performance at the current interaction. Positive values indicate overconfidence; negative values indicate underconfidence. Using this residual as the target directly encodes miscalibration magnitude and direction, and keeps the calibration task aligned with the interaction at timestep *t* while the KT head continues to predict correctness at *t + 1* under the existing shift that prevents leakage.

### Dual-Head Architecture

The model uses a single shared 2-layer LSTM backbone with two task-specific output heads. The **KT head** (linear → sigmoid) produces a probability of correctness on the next interaction, preserving the standard knowledge-tracing objective. The **calibration head** (linear → tanh) regresses the miscalibration residual in [−1, 1]. Sharing the LSTM encourages the hidden state to encode knowledge that is jointly useful for both performance prediction and metacognitive assessment, while separate heads allow each task to apply the appropriate output activation and loss function.

In [15]:
def sequences_from_df(frame: pd.DataFrame, num_skills: int):
    """List of (skill_ids, corrects, confidences, response_time_norm) per student."""
    seqs = []
    for _, g in frame.groupby("user_id", sort=False):
        g = g.sort_values("order_id", kind="mergesort")
        s = g["skill_id"].to_numpy(np.int64)
        y = g["correct"].to_numpy(np.float32)
        c = g["confidence"].to_numpy(np.float32)
        rt = g["response_time_norm"].to_numpy(np.float32)
        if len(s) > 0:
            seqs.append((s, y, c, rt))
    return seqs


class DKTSimpleDataset(Dataset):
    """Each timestep predicts next correctness (KT) and current calibration residual.

    x[t] = [skill one-hot, correctness, confidence, response_time_norm] with dim 2*K+2.
    y_kt[t] = correct_{t+1}; y_cal[t] = confidence_t - correct_t.
    """

    def __init__(self, seqs: list, num_skills: int, zero_confidence: bool = False):
        self.seqs = [(s, y, c, rt) for s, y, c, rt in seqs if len(s) >= 2]
        self.K = num_skills
        self.zero_confidence = zero_confidence

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        s, y, c, rt = self.seqs[idx]
        T = len(s)
        T_align = T - 1
        x = np.zeros((T_align, 2 * self.K + 2), dtype=np.float32)
        for t in range(T_align):
            k = int(s[t])
            x[t, k] = 1.0
            x[t, self.K + k] = float(y[t])
            x[t, 2 * self.K] = 0.0 if self.zero_confidence else float(c[t])
            x[t, 2 * self.K + 1] = float(rt[t])
        y_kt = y[1:T].astype(np.float32, copy=True)
        y_cal = (c[:T_align] - y[:T_align]).astype(np.float32, copy=True)
        return torch.from_numpy(x), torch.from_numpy(y_kt), torch.from_numpy(y_cal), T_align


class DKTBaselineDataset(Dataset):
    """Standard DKT input: skill one-hot + correctness only (2*K channels)."""

    def __init__(self, seqs: list, num_skills: int):
        self.seqs = [(s, y, c, rt) for s, y, c, rt in seqs if len(s) >= 2]
        self.K = num_skills

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        s, y, c, rt = self.seqs[idx]
        T = len(s)
        T_align = T - 1
        x = np.zeros((T_align, 2 * self.K), dtype=np.float32)
        for t in range(T_align):
            k = int(s[t])
            x[t, k] = 1.0
            x[t, self.K + k] = float(y[t])
        y_kt = y[1:T].astype(np.float32, copy=True)
        return torch.from_numpy(x), torch.from_numpy(y_kt), T_align


def collate_pad(batch):
    xs, y_kt, y_cal, lens = zip(*batch)
    B = len(batch)
    Tm = max(lens)
    D = xs[0].shape[1]
    xb = torch.zeros(B, Tm, D, dtype=torch.float32)
    y_kt_b = torch.zeros(B, Tm, dtype=torch.float32)
    y_cal_b = torch.zeros(B, Tm, dtype=torch.float32)
    mask = torch.zeros(B, Tm, dtype=torch.bool)
    for i, (x, yk, yc, L) in enumerate(batch):
        xb[i, :L] = x
        y_kt_b[i, :L] = yk
        y_cal_b[i, :L] = yc
        mask[i, :L] = True
    return xb, y_kt_b, y_cal_b, mask


def collate_pad_baseline(batch):
    xs, y_kt, lens = zip(*batch)
    B = len(batch)
    Tm = max(lens)
    D = xs[0].shape[1]
    xb = torch.zeros(B, Tm, D, dtype=torch.float32)
    y_kt_b = torch.zeros(B, Tm, dtype=torch.float32)
    mask = torch.zeros(B, Tm, dtype=torch.bool)
    for i, (x, yk, L) in enumerate(batch):
        xb[i, :L] = x
        y_kt_b[i, :L] = yk
        mask[i, :L] = True
    return xb, y_kt_b, mask


class JointKTCalibrationModel(nn.Module):
    def __init__(self, num_skills: int, hidden_size: int = 128):
        super().__init__()
        self.lstm = nn.LSTM(
            2 * num_skills + 2,
            hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
        )
        self.kt_head = nn.Linear(hidden_size, 1)
        self.cal_head = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor):
        h, _ = self.lstm(x)
        pred_kt = torch.sigmoid(self.kt_head(h).squeeze(-1))
        pred_cal = torch.tanh(self.cal_head(h).squeeze(-1))
        return pred_kt, pred_cal


class DKTBaseline(nn.Module):
    def __init__(self, num_skills, hidden_size=128):
        super().__init__()
        self.lstm = nn.LSTM(
            2 * num_skills,
            hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
        )
        self.kt_head = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor):
        h, _ = self.lstm(x)
        pred_kt = torch.sigmoid(self.kt_head(h).squeeze(-1))
        return pred_kt


train_seqs = sequences_from_df(train_df, num_skills)
val_seqs = sequences_from_df(val_df, num_skills)
test_seqs = sequences_from_df(test_df, num_skills)

train_loader = DataLoader(
    DKTSimpleDataset(train_seqs, num_skills),
    batch_size=32,
    shuffle=True,
    collate_fn=collate_pad,
)
val_loader = DataLoader(
    DKTSimpleDataset(val_seqs, num_skills),
    batch_size=64,
    shuffle=False,
    collate_fn=collate_pad,
)
test_loader = DataLoader(
    DKTSimpleDataset(test_seqs, num_skills),
    batch_size=64,
    shuffle=False,
    collate_fn=collate_pad,
)

baseline_train_loader = DataLoader(
    DKTBaselineDataset(train_seqs, num_skills),
    batch_size=32,
    shuffle=True,
    collate_fn=collate_pad_baseline,
)
baseline_val_loader = DataLoader(
    DKTBaselineDataset(val_seqs, num_skills),
    batch_size=64,
    shuffle=False,
    collate_fn=collate_pad_baseline,
)
baseline_test_loader = DataLoader(
    DKTBaselineDataset(test_seqs, num_skills),
    batch_size=64,
    shuffle=False,
    collate_fn=collate_pad_baseline,
)

model = JointKTCalibrationModel(num_skills).to(DEVICE)
dkt_baseline = DKTBaseline(num_skills).to(DEVICE)
print(model)
print(f"Joint input dimension: {2 * num_skills + 2}")
print(f"Joint parameters: {sum(p.numel() for p in model.parameters()):,}")
print(dkt_baseline)
print(f"Baseline input dimension: {2 * num_skills}")
print(f"Baseline parameters: {sum(p.numel() for p in dkt_baseline.parameters()):,}")

JointKTCalibrationModel(
  (lstm): LSTM(222, 128, num_layers=2, batch_first=True, dropout=0.3)
  (kt_head): Linear(in_features=128, out_features=1, bias=True)
  (cal_head): Linear(in_features=128, out_features=1, bias=True)
)
Joint input dimension: 222
Joint parameters: 312,578
DKTBaseline(
  (lstm): LSTM(220, 128, num_layers=2, batch_first=True, dropout=0.3)
  (kt_head): Linear(in_features=128, out_features=1, bias=True)
)
Baseline input dimension: 220
Baseline parameters: 311,425


## 7. Training Loop

**Combined loss** = BCE on KT predictions + λ × MSE on calibration predictions (valid timesteps only). Up to 40 epochs with early stopping (patience = 3) on validation KT loss. Uses Adam (lr=1e-3), `ReduceLROnPlateau` scheduler, and gradient clipping (max norm 1.0).

In [16]:
kt_criterion = nn.BCELoss(reduction="none")
cal_criterion = nn.MSELoss(reduction="none")
LAMBDA_CAL = 0.3
MAX_EPOCHS = 40
PATIENCE = 3


def compute_ece(y_true, y_prob, n_bins=10):
    """Expected Calibration Error using equal-width bins on predicted probabilities."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        if i == n_bins - 1:
            in_bin = (y_prob >= lo) & (y_prob <= hi)
        else:
            in_bin = (y_prob >= lo) & (y_prob < hi)
        prop = np.mean(in_bin)
        if prop > 0:
            acc = np.mean(y_true[in_bin])
            conf = np.mean(y_prob[in_bin])
            ece += prop * np.abs(acc - conf)
    return float(ece)


def run_joint_epoch(model, loader, optimizer, train_mode, lambda_cal=0.3):
    if train_mode:
        model.train()
    else:
        model.eval()
    total_kt, total_cal, ntok = 0.0, 0.0, 0
    for xb, y_kt, y_cal, mask in loader:
        xb = xb.to(DEVICE)
        y_kt = y_kt.to(DEVICE)
        y_cal = y_cal.to(DEVICE)
        mask = mask.to(DEVICE)
        if train_mode:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train_mode):
            pred_kt, pred_cal = model(xb)
            kt_mat = kt_criterion(pred_kt, y_kt)
            cal_mat = cal_criterion(pred_cal, y_cal)
            m = mask.float()
            denom = m.sum().clamp_min(1.0)
            kt_loss = (kt_mat * m).sum() / denom
            cal_loss = (cal_mat * m).sum() / denom
            loss = kt_loss + lambda_cal * cal_loss
        if train_mode:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        n = int(mask.sum().item())
        total_kt += float(kt_loss.item()) * n
        total_cal += float(cal_loss.item()) * n
        ntok += n
    denom = max(ntok, 1)
    return total_kt / denom, total_cal / denom


def run_baseline_epoch(model, loader, optimizer, train_mode):
    if train_mode:
        model.train()
    else:
        model.eval()
    total_kt, ntok = 0.0, 0
    for xb, y_kt, mask in loader:
        xb = xb.to(DEVICE)
        y_kt = y_kt.to(DEVICE)
        mask = mask.to(DEVICE)
        if train_mode:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train_mode):
            pred_kt = model(xb)
            kt_mat = kt_criterion(pred_kt, y_kt)
            m = mask.float()
            denom = m.sum().clamp_min(1.0)
            kt_loss = (kt_mat * m).sum() / denom
            loss = kt_loss
        if train_mode:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        n = int(mask.sum().item())
        total_kt += float(kt_loss.item()) * n
        ntok += n
    return total_kt / max(ntok, 1)


def train_joint_model(model, train_loader, val_loader, lambda_cal=0.3, label="Joint Model"):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2, verbose=True
    )
    best_val_kt_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        tr_kt, tr_cal = run_joint_epoch(model, train_loader, optimizer, True, lambda_cal)
        va_kt, va_cal = run_joint_epoch(model, val_loader, optimizer, False, lambda_cal)
        scheduler.step(va_kt)  # validation KT loss

        if va_kt < best_val_kt_loss:
            best_val_kt_loss = va_kt
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(f"[{label}] Epoch {epoch}/{MAX_EPOCHS}")
        print(f"  Train KT: {tr_kt:.4f}  Train Cal: {tr_cal:.4f}")
        print(f"  Val KT: {va_kt:.4f}  Val Cal: {va_cal:.4f}  Best Val KT: {best_val_kt_loss:.4f}")

        if epochs_without_improvement >= PATIENCE:
            print(f"  Early stopping after {epoch} epochs.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_state


def train_dkt_baseline(model, train_loader, val_loader, label="DKT Baseline"):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2, verbose=True
    )
    best_val_kt_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        tr_kt = run_baseline_epoch(model, train_loader, optimizer, True)
        va_kt = run_baseline_epoch(model, val_loader, optimizer, False)
        scheduler.step(va_kt)  # validation KT loss

        if va_kt < best_val_kt_loss:
            best_val_kt_loss = va_kt
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(f"[{label}] Epoch {epoch}/{MAX_EPOCHS}")
        print(f"  Train KT: {tr_kt:.4f}  Val KT: {va_kt:.4f}  Best Val KT: {best_val_kt_loss:.4f}")

        if epochs_without_improvement >= PATIENCE:
            print(f"  Early stopping after {epoch} epochs.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_state


def evaluate_joint(model, loader):
    model.eval()
    all_y_kt, all_p_kt, all_y_cal, all_p_cal = [], [], [], []
    with torch.no_grad():
        for xb, y_kt, y_cal, mask in loader:
            xb = xb.to(DEVICE)
            pred_kt, pred_cal = model(xb)
            pred_kt = pred_kt.cpu().numpy()
            pred_cal = pred_cal.cpu().numpy()
            m = mask.numpy()
            all_y_kt.append(y_kt.numpy()[m])
            all_p_kt.append(pred_kt[m])
            all_y_cal.append(y_cal.numpy()[m])
            all_p_cal.append(pred_cal[m])

    y_kt_true = np.concatenate(all_y_kt).astype(np.float64)
    y_kt_prob = np.concatenate(all_p_kt).astype(np.float64)
    y_cal_true = np.concatenate(all_y_cal).astype(np.float64)
    y_cal_pred = np.concatenate(all_p_cal).astype(np.float64)

    rmse = float(np.sqrt(mean_squared_error(y_kt_true, y_kt_prob)))
    cal_mse = float(mean_squared_error(y_cal_true, y_cal_pred))
    auc = float(roc_auc_score(y_kt_true, y_kt_prob)) if len(np.unique(y_kt_true)) >= 2 else float("nan")
    ece = compute_ece(y_kt_true, y_kt_prob)
    return {"auc": auc, "rmse": rmse, "cal_mse": cal_mse, "ece": ece}


def evaluate_baseline(model, loader):
    model.eval()
    all_y_kt, all_p_kt = [], []
    with torch.no_grad():
        for xb, y_kt, mask in loader:
            xb = xb.to(DEVICE)
            pred_kt = model(xb).cpu().numpy()
            m = mask.numpy()
            all_y_kt.append(y_kt.numpy()[m])
            all_p_kt.append(pred_kt[m])

    y_kt_true = np.concatenate(all_y_kt).astype(np.float64)
    y_kt_prob = np.concatenate(all_p_kt).astype(np.float64)
    rmse = float(np.sqrt(mean_squared_error(y_kt_true, y_kt_prob)))
    auc = float(roc_auc_score(y_kt_true, y_kt_prob)) if len(np.unique(y_kt_true)) >= 2 else float("nan")
    return {"auc": auc, "rmse": rmse}


print("Training Joint KT + Calibration model...")
best_state = train_joint_model(model, train_loader, val_loader, lambda_cal=LAMBDA_CAL)

Training Joint KT + Calibration model...
[Joint Model] Epoch 1/40
  Train KT: 0.5519  Train Cal: 0.0472
  Val KT: 0.4705  Val Cal: 0.0445  Best Val KT: 0.4705
[Joint Model] Epoch 2/40
  Train KT: 0.4696  Train Cal: 0.0378
  Val KT: 0.4265  Val Cal: 0.0343  Best Val KT: 0.4265
[Joint Model] Epoch 3/40
  Train KT: 0.4313  Train Cal: 0.0340
  Val KT: 0.3874  Val Cal: 0.0317  Best Val KT: 0.3874
[Joint Model] Epoch 4/40
  Train KT: 0.4067  Train Cal: 0.0324
  Val KT: 0.3711  Val Cal: 0.0294  Best Val KT: 0.3711
[Joint Model] Epoch 5/40
  Train KT: 0.3934  Train Cal: 0.0305
  Val KT: 0.3631  Val Cal: 0.0290  Best Val KT: 0.3631
[Joint Model] Epoch 6/40
  Train KT: 0.3862  Train Cal: 0.0296
  Val KT: 0.3567  Val Cal: 0.0283  Best Val KT: 0.3567
[Joint Model] Epoch 7/40
  Train KT: 0.3810  Train Cal: 0.0292
  Val KT: 0.3546  Val Cal: 0.0272  Best Val KT: 0.3546
[Joint Model] Epoch 8/40
  Train KT: 0.3780  Train Cal: 0.0279
  Val KT: 0.3531  Val Cal: 0.0263  Best Val KT: 0.3531
[Joint Model] E

## 7b. DKT Baseline Training

Fair baseline using only skill one-hot and correctness (2×K input), trained under identical conditions as the joint model.

In [17]:
print("Training DKT Baseline...")
dkt_baseline = DKTBaseline(num_skills).to(DEVICE)
baseline_state = train_dkt_baseline(dkt_baseline, baseline_train_loader, baseline_val_loader)
baseline_metrics = evaluate_baseline(dkt_baseline, baseline_test_loader)

print(f"DKT Baseline - AUC: {baseline_metrics['auc']:.4f}")
print(f"DKT Baseline - RMSE: {baseline_metrics['rmse']:.4f}")

Training DKT Baseline...
[DKT Baseline] Epoch 1/40
  Train KT: 0.5817  Val KT: 0.5093  Best Val KT: 0.5093
[DKT Baseline] Epoch 2/40
  Train KT: 0.4901  Val KT: 0.4337  Best Val KT: 0.4337
[DKT Baseline] Epoch 3/40
  Train KT: 0.4457  Val KT: 0.3937  Best Val KT: 0.3937
[DKT Baseline] Epoch 4/40
  Train KT: 0.4103  Val KT: 0.3750  Best Val KT: 0.3750
[DKT Baseline] Epoch 5/40
  Train KT: 0.3943  Val KT: 0.3644  Best Val KT: 0.3644
[DKT Baseline] Epoch 6/40
  Train KT: 0.3886  Val KT: 0.3592  Best Val KT: 0.3592
[DKT Baseline] Epoch 7/40
  Train KT: 0.3825  Val KT: 0.3563  Best Val KT: 0.3563
[DKT Baseline] Epoch 8/40
  Train KT: 0.3789  Val KT: 0.3549  Best Val KT: 0.3549
[DKT Baseline] Epoch 9/40
  Train KT: 0.3773  Val KT: 0.3516  Best Val KT: 0.3516
[DKT Baseline] Epoch 10/40
  Train KT: 0.3749  Val KT: 0.3501  Best Val KT: 0.3501
[DKT Baseline] Epoch 11/40
  Train KT: 0.3731  Val KT: 0.3498  Best Val KT: 0.3498
[DKT Baseline] Epoch 12/40
  Train KT: 0.3720  Val KT: 0.3498  Best Val

## 8. Evaluation

**KT AUC** and **KT RMSE** on next-step correctness; **Calibration MSE** on the miscalibration residual at the interaction level.

In [18]:
joint_metrics = evaluate_joint(model, test_loader)
auc = joint_metrics["auc"]
rmse = joint_metrics["rmse"]
cal_mse = joint_metrics["cal_mse"]
joint_ece = joint_metrics["ece"]

print(f"Joint Model - AUC: {auc:.4f}")
print(f"Joint Model - RMSE: {rmse:.4f}")
print(f"Joint Model - Calibration MSE: {cal_mse:.4f}")
print(f"Joint Model - ECE: {joint_ece:.4f}")

Joint Model - AUC: 0.9171
Joint Model - RMSE: 0.3161
Joint Model - Calibration MSE: 0.0025
Joint Model - ECE: 0.0051


## 9. Miscalibration Detector

The detector compares self-reported **confidence** to observed **correctness** at each interaction.

- **Overconfidence** — confidence substantially exceeds performance (high confidence despite incorrect responses).
- **Underconfidence** — confidence substantially falls below performance (low confidence despite correct responses).
- **Calibrated** — confidence approximately matches performance (residual gap within ±0.3).

In [19]:
def detect_miscalibration(confidence, correctness, threshold=0.3):
    """Label each interaction as overconfident, underconfident, or calibrated."""
    calibration_gap = confidence - correctness
    if np.ndim(calibration_gap) == 0:
        if calibration_gap >= threshold:
            return "overconfident"
        if calibration_gap <= -threshold:
            return "underconfident"
        return "calibrated"

    labels = np.full(calibration_gap.shape, "calibrated", dtype=object)
    labels[calibration_gap >= threshold] = "overconfident"
    labels[calibration_gap <= -threshold] = "underconfident"
    return labels

## 10. Intervention Selector

Given estimated **knowledge** and a **miscalibration label**, the selector assigns one of four intervention types:

- **Targeted Explanation** — delivers concept-specific instruction when a low-knowledge learner is overconfident.
- **Confidence Recalibration Prompt** — prompts a high-knowledge but overconfident learner to reassess certainty.
- **Reflective Question** — encourages metacognitive reflection; used for underconfident low-knowledge learners and calibrated learners.
- **Counterfactual Reasoning Prompt** — asks a high-knowledge underconfident learner to consider alternative solution paths that justify their competence.

In [20]:
def select_intervention(knowledge_score, miscalibration_label):
    """Map knowledge estimate and miscalibration label to an intervention type."""
    if np.ndim(knowledge_score) == 0 and isinstance(miscalibration_label, str):
        if miscalibration_label == "calibrated":
            return "Reflective Question"
        if miscalibration_label == "overconfident":
            if knowledge_score < 0.5:
                return "Targeted Explanation"
            return "Confidence Recalibration Prompt"
        if miscalibration_label == "underconfident":
            if knowledge_score < 0.5:
                return "Reflective Question"
            return "Counterfactual Reasoning Prompt"
        raise ValueError(f"Unknown miscalibration label: {miscalibration_label!r}")

    knowledge_score = np.asarray(knowledge_score, dtype=np.float64)
    labels = np.asarray(miscalibration_label, dtype=object)
    interventions = np.empty(labels.shape, dtype=object)

    calibrated = labels == "calibrated"
    over = labels == "overconfident"
    under = labels == "underconfident"

    interventions[calibrated] = "Reflective Question"
    interventions[over & (knowledge_score < 0.5)] = "Targeted Explanation"
    interventions[over & (knowledge_score >= 0.5)] = "Confidence Recalibration Prompt"
    interventions[under & (knowledge_score < 0.5)] = "Reflective Question"
    interventions[under & (knowledge_score >= 0.5)] = "Counterfactual Reasoning Prompt"
    return interventions

## 11. Test Set Analysis

Apply the miscalibration detector and intervention selector to every valid test interaction, using observed confidence and correctness together with the model's predicted KT probability as the knowledge score.

In [21]:
all_confidence, all_correctness, all_knowledge = [], [], []

model.eval()
with torch.no_grad():
    for xb, y_kt, y_cal, mask in test_loader:
        xb = xb.to(DEVICE)
        pred_kt, _ = model(xb)
        conf = xb[:, :, 2 * num_skills].cpu().numpy()
        correct = conf - y_cal.numpy()
        pred_kt = pred_kt.cpu().numpy()
        m = mask.numpy()
        all_confidence.append(conf[m])
        all_correctness.append(correct[m])
        all_knowledge.append(pred_kt[m])

confidence = np.concatenate(all_confidence).astype(np.float64)
correctness = np.concatenate(all_correctness).astype(np.float64)
knowledge_score = np.concatenate(all_knowledge).astype(np.float64)

miscal_labels = detect_miscalibration(confidence, correctness, threshold=0.3)
interventions = select_intervention(knowledge_score, miscal_labels)

n = len(miscal_labels)
over_pct = 100.0 * np.mean(miscal_labels == "overconfident")
under_pct = 100.0 * np.mean(miscal_labels == "underconfident")
cal_pct = 100.0 * np.mean(miscal_labels == "calibrated")
misc_pct = 100.0 * np.mean(miscal_labels != "calibrated")

intervention_types = [
    "Targeted Explanation",
    "Confidence Recalibration Prompt",
    "Reflective Question",
    "Counterfactual Reasoning Prompt",
]
intervention_pcts = {name: 100.0 * np.mean(interventions == name) for name in intervention_types}

print(f"Overall Miscalibration Rate: {misc_pct:.2f}%\n")
print(f"Overconfident: {over_pct:.2f}%")
print(f"Underconfident: {under_pct:.2f}%")
print(f"Calibrated: {cal_pct:.2f}%\n")
print("Intervention Distribution:")
for name in intervention_types:
    print(f"{name}: {intervention_pcts[name]:.2f}%")

Overall Miscalibration Rate: 21.18%

Overconfident: 6.36%
Underconfident: 14.82%
Calibrated: 78.82%

Intervention Distribution:
Targeted Explanation: 4.27%
Confidence Recalibration Prompt: 2.09%
Reflective Question: 78.99%
Counterfactual Reasoning Prompt: 14.65%


## 12. Expected Calibration Error (ECE)

ECE measures alignment between predicted KT probabilities and observed correctness. Predictions are grouped into 10 equal-width bins on \[0, 1\]; for each bin, the weighted absolute gap between mean confidence and mean accuracy contributes to the final score.

In [22]:
# ECE computed in evaluate_joint() during Section 8 evaluation.
print(f"Joint Model - ECE: {joint_ece:.4f}")

Joint Model - ECE: 0.0051


## 14. Ablation Study

### A. Lambda Sweep

Vary the calibration loss weight λ ∈ {0.1, 0.3, 0.5} while holding all other training settings fixed.

### B. No-Confidence Ablation

Train a joint model with confidence input forced to zero at every timestep, isolating the contribution of the confidence channel.

In [ ]:
# --- A. Lambda Sweep ---
lambda_results = {}
for lam in [0.1, 0.3, 0.5]:
    lam_model = JointKTCalibrationModel(num_skills).to(DEVICE)
    train_joint_model(
        lam_model, train_loader, val_loader,
        lambda_cal=lam, label=f"Lambda={lam}",
    )
    lambda_results[lam] = evaluate_joint(lam_model, test_loader)

lambda_table = """| λ_cal | AUC ↑ | RMSE ↓ | Calibration MSE ↓ | ECE ↓ |
|-------|-------|--------|-------------------|-------|
"""
for lam in [0.1, 0.3, 0.5]:
    m = lambda_results[lam]
    lambda_table += f"| {lam} | {m['auc']:.4f} | {m['rmse']:.4f} | {m['cal_mse']:.4f} | {m['ece']:.4f} |\n"

print("Lambda Sweep Results:")
display(Markdown(lambda_table))

# --- B. No-Confidence Ablation ---
noconf_train_loader = DataLoader(
    DKTSimpleDataset(train_seqs, num_skills, zero_confidence=True),
    batch_size=32, shuffle=True, collate_fn=collate_pad,
)
noconf_val_loader = DataLoader(
    DKTSimpleDataset(val_seqs, num_skills, zero_confidence=True),
    batch_size=64, shuffle=False, collate_fn=collate_pad,
)
noconf_test_loader = DataLoader(
    DKTSimpleDataset(test_seqs, num_skills, zero_confidence=True),
    batch_size=64, shuffle=False, collate_fn=collate_pad,
)

noconf_model = JointKTCalibrationModel(num_skills).to(DEVICE)
train_joint_model(
    noconf_model, noconf_train_loader, noconf_val_loader,
    lambda_cal=LAMBDA_CAL, label="No Confidence",
)
noconf_metrics = evaluate_joint(noconf_model, noconf_test_loader)
full_metrics = joint_metrics

noconf_table = f"""| Model | AUC ↑ | RMSE ↓ | Calibration MSE ↓ | ECE ↓ |
|-------|-------|--------|-------------------|-------|
| Joint KT+Cal (No Confidence) | {noconf_metrics['auc']:.4f} | {noconf_metrics['rmse']:.4f} | {noconf_metrics['cal_mse']:.4f} | {noconf_metrics['ece']:.4f} |
| Joint KT+Cal (Full) | {full_metrics['auc']:.4f} | {full_metrics['rmse']:.4f} | {full_metrics['cal_mse']:.4f} | {full_metrics['ece']:.4f} |
"""

print("\nNo-Confidence Ablation:")
display(Markdown(noconf_table))

[Lambda=0.1] Epoch 1/40
  Train KT: 0.5541  Train Cal: 0.0479
  Val KT: 0.4625  Val Cal: 0.0401  Best Val KT: 0.4625
[Lambda=0.1] Epoch 2/40
  Train KT: 0.4677  Train Cal: 0.0385
  Val KT: 0.4169  Val Cal: 0.0333  Best Val KT: 0.4169
[Lambda=0.1] Epoch 3/40
  Train KT: 0.4266  Train Cal: 0.0336
  Val KT: 0.3843  Val Cal: 0.0303  Best Val KT: 0.3843
[Lambda=0.1] Epoch 4/40
  Train KT: 0.4059  Train Cal: 0.0315
  Val KT: 0.3687  Val Cal: 0.0296  Best Val KT: 0.3687
[Lambda=0.1] Epoch 5/40
  Train KT: 0.3913  Train Cal: 0.0312
  Val KT: 0.3657  Val Cal: 0.0302  Best Val KT: 0.3657
[Lambda=0.1] Epoch 6/40
  Train KT: 0.3883  Train Cal: 0.0303
  Val KT: 0.3617  Val Cal: 0.0291  Best Val KT: 0.3617
[Lambda=0.1] Epoch 7/40
  Train KT: 0.3814  Train Cal: 0.0296
  Val KT: 0.3545  Val Cal: 0.0286  Best Val KT: 0.3545
[Lambda=0.1] Epoch 8/40
  Train KT: 0.3777  Train Cal: 0.0315
  Val KT: 0.3531  Val Cal: 0.0290  Best Val KT: 0.3531
[Lambda=0.1] Epoch 9/40
  Train KT: 0.3770  Train Cal: 0.0290
  

## 15. Per-Skill Miscalibration

Interactions where |confidence − correct| ≥ 0.3 are flagged as miscalibrated. The table below lists the 10 skills with the highest miscalibration rates on the test set.

In [24]:
test_eval = test_df.copy()
test_eval["miscalibrated"] = (test_eval["confidence"] - test_eval["correct"]).abs() >= 0.3

skill_misc = (
    test_eval.groupby("skill_name")["miscalibrated"]
    .agg(miscalibration_rate="mean", n_interactions="count")
    .sort_values("miscalibration_rate", ascending=False)
    .head(10)
    .reset_index()
)

skill_misc["miscalibration_rate"] = (100.0 * skill_misc["miscalibration_rate"]).round(2)
skill_misc.columns = ["Skill", "Miscalibration Rate (%)", "N Interactions"]

skill_table = """| Skill | Miscalibration Rate (%) | N Interactions |
|-------|-------------------------|----------------|
"""
for _, row in skill_misc.iterrows():
    skill_table += f"| {row['Skill']} | {row['Miscalibration Rate (%)']} | {row['N Interactions']} |\n"

print("Top 10 Skills by Miscalibration Rate:")
display(Markdown(skill_table))

Top 10 Skills by Miscalibration Rate:


| Skill | Miscalibration Rate (%) | N Interactions |
|-------|-------------------------|----------------|
| Solving Systems of Linear Equations by Graphing | 100.0 | 1 |
| Solving Systems of Linear Equations | 62.5 | 8 |
| Intercept | 53.85 | 13 |
| Algebraic Simplification | 50.0 | 2 |
| Perimeter of a Polygon | 38.1 | 42 |
| Multiplication Whole Numbers | 33.33 | 27 |
| Percents | 33.33 | 18 |
| Parts of a Polyomial, Terms, Coefficient, Monomial, Exponent, Variable | 31.48 | 54 |
| Linear Equations | 30.77 | 13 |
| Surface Area Cylinder | 30.43 | 46 |


## 13. Results Table

In [ ]:
results_table = f"""| Model | AUC ↑ | RMSE ↓ | Calibration MSE ↓ | ECE ↓ |
|---------|---------|---------|---------|---------|
| DKT Baseline | {baseline_metrics['auc']:.4f} | {baseline_metrics['rmse']:.4f} | N/A | N/A |
| Joint KT + Calibration | {auc:.4f} | {rmse:.4f} | {cal_mse:.4f} | {joint_ece:.4f} |

The joint model maintains or slightly improves predictive performance relative to the DKT baseline while introducing an explicit calibration modeling pathway. The baseline DKT cannot produce calibration-specific metrics such as Calibration MSE or ECE because it predicts only next-step correctness. ECE provides an additional evaluation dimension focused on metacognitive alignment—quantifying how well predicted probabilities match observed outcomes across confidence bins—complementing the discrimination metrics captured by AUC and RMSE."""

display(Markdown(results_table))

| Model | AUC ↑ | RMSE ↓ | Calibration MSE ↓ | ECE ↓ |
|---------|---------|---------|---------|---------|
| DKT Baseline | 0.9133 | 0.3193 | N/A | N/A |
| Joint KT + Calibration | 0.9138 | 0.3191 | 0.0021 | 0.0053 |

The joint model maintains or slightly improves predictive performance relative to the DKT baseline while introducing an explicit calibration modeling pathway. The baseline DKT cannot produce calibration-specific metrics such as Calibration MSE or ECE because it predicts only next-step correctness. ECE provides an additional evaluation dimension focused on metacognitive alignment—quantifying how well predicted probabilities match observed outcomes across confidence bins—complementing the discrimination metrics captured by AUC and RMSE.